In [1]:
import torch
from transformers import GPT2LMHeadModel
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast.from_pretrained("skt/kogpt2-base-v2",
bos_token='</s>', eos_token='</s>', unk_token='<unk>',
pad_token='<pad>', mask_token='<mask>')

model = GPT2LMHeadModel.from_pretrained('skt/kogpt2-base-v2')

c:\Users\Playdata\AppData\Local\miniconda3\envs\new_01\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 149/149 [00:00<00:00, 21414.18it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
# 앵무새병... 1등 단어만 맹목적으로 고르는 Greedy 특성상 한번 특정문구패턴이 1등 확률을차지하지 시작하면 반복됨
prompt_text = "인공지능이 인간을 대처할 수 있을까?"
input_ids = tokenizer.encode(prompt_text,return_tensors='pt')

with torch.no_grad():
    greedy_output =  model.generate(
        input_ids,
        max_length = 50,
        pad_token_id = tokenizer.pad_token_id
    )
    
generated_text = tokenizer.decode(greedy_output[0],skip_special_tokens=True)    
print(generated_text)

인공지능이 인간을 대처할 수 있을까?"라며 "그런데도 우리는 인공지능이 인간을 어떻게 대처할 수 있는지 알지 못한다"고 말했다.
그는 "인공지능은 인간의 행동을 예측하고 통제할 수 있는 능력을 갖고


In [3]:
# Sampling : 무조건 1등 단어만 뽑는게 아니라 주사위 굴리듯이 확률에 기반하여 가끔은 2등 3등 단어도 뽑아주는 랜덤성 기법 도입
with torch.no_grad():
    sample_output =  model.generate(
        input_ids,
        max_length = 50,
        do_sample=True,  # 주사위 굴리기
        top_k = 50,  # 상위50개중에서 주사위 굴리기
        top_p = 0.9, # 누적확률 90% 안에서
        temperature=0.8, # 1이면 미친 창의성  0.1 , 0 완전 안전하게 확률에 기반(창작을 거의 안함)
        repetition_penalty = 1.2,  # 했던 말 또하면 벌점
        pad_token_id = tokenizer.pad_token_id
    )
creative_text = tokenizer.decode(sample_output[0], skip_special_tokens=True)    
print(creative_text)

인공지능이 인간을 대처할 수 있을까?"라고 되물었다.
이어 "어쩌면 '인간보다 로봇을 더 두려워하는 건가?'라고 생각한다면 오산이다. 그건 로봇의 힘을 믿는 것 때문이다"라고 덧붙였다.
그는 자신의 블로그


In [6]:
# 파인튜닝
from torch.optim import AdamW
train_text = [
    '질문 : 자연어 처리가 뭔가요? 답변 : 컴퓨터가 인간의 언어를 이해하고 분석하는 인공지능 기술입니다.',
    '질문 : 오늘 기분이 어때? 답변 : 저는 인공지능이라 기분을 느낄수 없지만, 당신과 대화해서 기뻐요'
]
inputs = tokenizer(train_text,padding=True, truncation=True, return_tensors='pt')
input_ids = inputs['input_ids']
attention_mask = inputs['attention_mask']
optimizer = AdamW(model.parameters(), lr=5e-5)
model.train()
epochs = 10
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(input_ids=input_ids,attention_mask=attention_mask,labels=input_ids)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    print(f'epoch {epoch+1}/{epochs} loss = {loss.item()}')
model.eval()
test_prompt = '질문:자연어 처리가 뭔가요? 답변:'    
test_ids = tokenizer.encode(test_prompt,return_tensors='pt')
with torch.no_grad():
    test_output = model.generate(
        test_ids,
        max_length = 40,
        pad_token_id = tokenizer.pad_token_id
    )
print( tokenizer.decode(test_output[0],skip_special_tokens=True)     )

epoch 1/10 loss = 0.2497389167547226
epoch 2/10 loss = 0.08627458661794662
epoch 3/10 loss = 0.3031516671180725
epoch 4/10 loss = 0.13495390117168427
epoch 5/10 loss = 0.15051834285259247
epoch 6/10 loss = 0.2122107595205307
epoch 7/10 loss = 0.0865233764052391
epoch 8/10 loss = 0.05193311348557472
epoch 9/10 loss = 0.047784168273210526
epoch 10/10 loss = 0.05719762295484543
질문:자연어 처리가 뭔가요? 답변: 컴퓨터가 인간의 언어를 이해하고 분석하는 인공지능 기술입니다.


In [7]:
model.eval()
test_prompt = '질문:철수가 영희를 괴롭히면 어떻게 할까요? 답변:'
test_ids = tokenizer.encode(test_prompt,return_tensors='pt')
with torch.no_grad():
    test_output = model.generate(
        test_ids,
        max_length = 40,
        temperature=0.8,
        pad_token_id = tokenizer.pad_token_id
    )
print( tokenizer.decode(test_output[0],skip_special_tokens=True)     )

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


질문:철수가 영희를 괴롭히면 어떻게 할까요? 답변: 컴퓨터가 인간의 언어를 이해하고 분석하는 인공지능 기술입니다.
